In [3]:
import os

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import plotly.graph_objects as go
from dotenv import load_dotenv

from gene_regulatory.grn_path_scoring import score_paths, setup_graph, up_or_down
from utils.vis_utils import change_graph_ids, colour_path

load_dotenv(override=True)
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
filter_graph, scored_graph, ho_scored_graph, virus_nodes = setup_graph(5)
all_info = score_paths(filter_graph, scored_graph, ho_scored_graph, virus_nodes)

Removed 6731 nodes: 1,12,14,16,19
Found 4 replicates across 6 timepoints.
Loading existing batch corrected data
Removed 5548/18226 nodes
5548 nodes without exp, 0 are virus


Scoring random 4 graph: 100%|██████████| 12357/12357 [00:03<00:00, 3162.01nodes/s]


Cases with |r| > 0.522 survive BH correction
Removing 65627 edges out of 96119 total
Better: virus, Score virus 0.78, Score human 0.52: 43740570 -> 6772 -> 3665 -> 567
Better: virus, Score virus 0.87, Score human 0.58: 43740571 -> 4841 -> 9513 -> 2495
Better: virus, Score virus 0.62, Score human 0.59: 43740570 -> 7157 -> 7022 -> 7178
Better: virus, Score virus 0.58, Score human 0.48: 43740571 -> 1999 -> 4609 -> 71
Better: virus, Score virus 0.63, Score human 0.50: 43740571 -> 4904 -> 7157 -> 6303
Better: virus, Score virus 0.68, Score human 0.53: 43740571 -> 5925 -> 6667 -> 7316
Better: virus, Score virus 0.59, Score human 0.44: 43740571 -> 5925 -> 7020 -> 7114
Better: virus, Score virus 0.55, Score human 0.47: 43740571 -> 22926 -> 4601 -> 6170
Better: virus, Score virus 0.70, Score human 0.59: 43740571 -> 5610 -> 6772 -> 1958 -> 8519
Better: virus, Score virus 0.63, Score human 0.62: 43740575 -> 4088 -> 3727 -> 6929 -> 226
Better: virus, Score virus 0.69, Score human 0.50: 43740575 ->

In [5]:
all_info[0].keys()

dict_keys(['node_name', 'alert_graph', 'best', 'best_v_score', 'best_h_score', 'path_lengths', 'best_path', 'bg_path_scores'])

In [6]:
sizes = []
for i in range(len(all_info)):
    sizes.append(len(list(all_info[i]["alert_graph"].nodes())))

print(sizes)
print(np.argmax(sizes))

[18, 45, 51, 15, 55, 66, 42, 21, 17, 46, 52, 18, 22, 17, 84, 44, 51, 49, 19, 20, 42, 84, 82, 9, 17, 45, 51, 28, 17, 46, 84, 16, 28, 16, 47, 20, 22, 28, 76, 54]
14


In [7]:
selected_alert = 0

G = all_info[selected_alert]["alert_graph"]
colour_path(G, all_info[selected_alert]["best_path"])
G = change_graph_ids(G)

pos = nx.spring_layout(G, weight=None)
nx.is_directed_acyclic_graph(G)

True

## Set up graph

In [8]:
weights = [d["weight"] for _, _, d in G.edges(data=True)]
w_min, w_max = 0, max(weights)


def normalize_opacity(w, lo=0.3, hi=1.0):
    if w_max == w_min:
        return hi
    return lo + (w - w_min) / (w_max - w_min) * (hi - lo)

In [9]:
# EDGES
edge_annotations = []
for u, v, data in G.edges(data=True):
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    edge_annotations.append(
        {
            "ax": x0,
            "ay": y0,
            "x": x1,
            "y": y1,
            "xref": "x",
            "yref": "y",
            "axref": "x",
            "ayref": "y",
            "showarrow": True,
            "arrowhead": 3,
            "arrowsize": 1.5,
            "arrowwidth": 1.5 + data["weight"] * 0.3,  # optional: thicker = heavier
            "arrowcolor": data["color"],
            "opacity": normalize_opacity(abs(data["real_weight"])),
            "standoff": 15,
            "startstandoff": 15,
        }
    )

weight_annotations = []
for u, v, data in G.edges(data=True):
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    mx, my = (x0 + x1) / 2, (y0 + y1) / 2
    weight_annotations.append(
        {
            "x": mx,
            "y": my,
            "xref": "x",
            "yref": "y",
            "text": f"{data['real_weight']:.2f}",
            "showarrow": False,
            "font": {"size": 11, "color": "black"},
            "bgcolor": "rgba(255,255,255,0.7)",  # readability over crossing edges
            "yshift": 8,  # nudges label "above" the line
        }
    )

if len(list(G.nodes())) > 30:
    all_annotations = edge_annotations
else:
    all_annotations = edge_annotations + weight_annotations

In [10]:
# NODES
node_x = [pos[n][0] for n in G.nodes()]
node_y = [pos[n][1] for n in G.nodes()]
node_colors = [G.nodes[n]["color"] for n in G.nodes()]
node_text = list(G.nodes())

node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode="markers+text",
    text=node_text,
    textposition="top center",
    marker={
        "size": 20,
        "color": node_colors,  # direct color values work if they're valid CSS colors
        "line": {"width": 2, "color": "black"},
    },
    hoverinfo="text",
)

## Show graph

In [11]:
fig = go.Figure(data=[node_trace])
fig.update_layout(
    annotations=all_annotations,
    showlegend=False,
    margin={"l": 20, "r": 20, "t": 20, "b": 20},
    xaxis={"showgrid": False, "zeroline": False, "showticklabels": False},
    yaxis={"showgrid": False, "zeroline": False, "showticklabels": False},
)
fig.show()

In [13]:
sentence = "As a result of "
for i, node in enumerate(all_info[selected_alert]["best_path"]):
    if scored_graph.nodes[node]["is_virus"]:
        sentence += f"{scored_graph.nodes[node]['name']} activity "
        continue
    slope = up_or_down(scored_graph.nodes[node]["time"], scored_graph.nodes[node]["timepoints"])
    cont = "is " if i < 2 else "to be "
    sentence += f"{scored_graph.nodes[node]['name']} {cont}"
    if slope > 0:
        sentence += f"upregulated ({slope:.2f}). "
    elif slope < 0:
        sentence += f"downregulated ({slope:.2f}). "

    if i < len(all_info[selected_alert]["best_path"]) - 1:
        sentence += "Which causes "

print(sentence)

As a result of E (v) activity STAT1 is upregulated (63.87). Which causes IRF7 to be upregulated (6.26). Which causes B2M to be upregulated (772.42). 
